In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim

In [10]:
# Constants
num_epochs = 1000
batch_size = 32
learning_rate = .2

criterion = nn.MSELoss()

RANDOM_STATE = 1082
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [1]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
    exit()

In [12]:
# Data Preprocessing

df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

if y.dtype == object or y.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train (benign only) and test (all)
X_train = X_scaled[y == 0]
X_test = X_scaled
y_test = y

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [13]:
# Define the Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=8):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

input_dim = X_train.shape[1]
encoding_dim = 8
autoencoder = Autoencoder(input_dim, encoding_dim)

In [14]:
# Training
optimizer = optim.Adam(autoencoder.parameters(), lr=learning_rate)
train_loader = torch.utils.data.DataLoader(X_train_tensor, batch_size=batch_size, shuffle=True)

autoencoder.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        output = autoencoder(batch)
        loss = criterion(output, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(X_train_tensor)
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.6f}")

Epoch 1/1000, Loss: 0.223972
Epoch 10/1000, Loss: 0.211803
Epoch 20/1000, Loss: 0.212684
Epoch 30/1000, Loss: 0.212133
Epoch 40/1000, Loss: 0.213438
Epoch 50/1000, Loss: 0.213707
Epoch 60/1000, Loss: 0.212080
Epoch 70/1000, Loss: 0.212644
Epoch 80/1000, Loss: 0.212012
Epoch 90/1000, Loss: 0.213557
Epoch 100/1000, Loss: 0.212024
Epoch 110/1000, Loss: 0.212690
Epoch 120/1000, Loss: 0.211632
Epoch 130/1000, Loss: 0.212562
Epoch 140/1000, Loss: 0.211876
Epoch 150/1000, Loss: 0.211749
Epoch 160/1000, Loss: 0.212127
Epoch 170/1000, Loss: 0.212481
Epoch 180/1000, Loss: 0.212012
Epoch 190/1000, Loss: 0.213318
Epoch 200/1000, Loss: 0.213220
Epoch 210/1000, Loss: 0.211530
Epoch 220/1000, Loss: 0.211876
Epoch 230/1000, Loss: 0.211768
Epoch 240/1000, Loss: 0.211527
Epoch 250/1000, Loss: 0.213095
Epoch 260/1000, Loss: 0.212858
Epoch 270/1000, Loss: 0.212143
Epoch 280/1000, Loss: 0.211964
Epoch 290/1000, Loss: 0.211975
Epoch 300/1000, Loss: 0.212538
Epoch 310/1000, Loss: 0.213309
Epoch 320/1000, Los

In [15]:
# Evaluation
autoencoder.eval()
with torch.no_grad():
    reconstructed = autoencoder(X_test_tensor)
    mse = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1).cpu().numpy()

# Threshold: 95th percentile of benign errors
threshold = np.percentile(mse[y == 0], 95)
y_pred = (mse > threshold).astype(int)


In [16]:
# Results
print(f"Autoencoder Threshold: {threshold:.4f}")
print(f"Model Accuracy on Test Set: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Autoencoder Threshold: 0.9426
Model Accuracy on Test Set: 95.44%
Confusion Matrix:
[[2967  157]
 [   8  490]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      3124
           1       0.76      0.98      0.86       498

    accuracy                           0.95      3622
   macro avg       0.88      0.97      0.91      3622
weighted avg       0.96      0.95      0.96      3622

